# ADOFAI Chart Generation Training (v1)

Train an AI model to generate ADOFAI (A Dance of Fire and Ice) charts from audio.

**Status**: Foundation training pipeline. Uses proof-of-concept LSTM model (Whisper integration TODO).

**Requirements**:
- ADOFAI charts in Google Drive (`level.adofai` + audio)
- GPU runtime (free tier: ~12 hours, Pro: longer)
- At least 100 charts recommended for initial quality

**Note**: Pretrained osu! weights are NOT compatible with ADOFAI. This trains from scratch.

## 1. Check GPU Runtime

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected. Training will be very slow on CPU.")
    print("   Go to Runtime → Change runtime type → GPU")

## 2. Configuration

In [ ]:
# === Configuration ===

# GitHub repo and branch
REPO_URL = "https://github.com/Tiller431/Mapperatorinator-ADOFAI.git"
BRANCH = "cursor/adofai-foundation-5317"  # Change to 'main' once PR is merged

# Google Drive paths (CHANGE THESE to match your Drive layout)
DATA_DIR = "/content/drive/MyDrive/adofai-dataset/charts-top100"  # Folder with chart subdirs
OUTPUT_DIR = "/content/drive/MyDrive/adofai-checkpoints"          # Where to save checkpoints

# Training settings
BATCH_SIZE = 4        # Adjust based on GPU memory (lower if OOM)
LEARNING_RATE = 1e-4
EPOCHS = 50           # For full training
MAX_SAMPLES = None    # None = use all charts; set to number for limited run

# Smoke test settings (quick validation)
SMOKE_MODE = False    # Set to True for quick test

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✓ Configuration set")
print(f"  Data dir: {DATA_DIR}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Device: {DEVICE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")

## 3. Mount Google Drive

Your ADOFAI charts should be organized in Drive like:
```
MyDrive/adofai-dataset/charts-top100/
  1234567__ChartName/
    level.adofai
    song.ogg
  9876543__AnotherChart/
    level.adofai
    audio.mp3
  ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify data directory exists
import os
if os.path.exists(DATA_DIR):
    chart_dirs = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"✓ Found {len(chart_dirs)} chart directories in {DATA_DIR}")
    if len(chart_dirs) > 0:
        print(f"  Examples: {chart_dirs[:3]}")
else:
    print(f"⚠️  Data directory not found: {DATA_DIR}")
    print(f"   Please create it and add ADOFAI chart folders.")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory ready: {OUTPUT_DIR}")

## 4. Clone Repository and Install Dependencies

In [ ]:
# Clone repo
!rm -rf Mapperatorinator-ADOFAI  # Remove if exists
!git clone -b {BRANCH} {REPO_URL}
%cd Mapperatorinator-ADOFAI

print(f"\n✓ Cloned {REPO_URL} (branch: {BRANCH})")

In [ ]:
# Install minimal dependencies for training
# Note: Skip heavy deps not needed for ADOFAI training

print("Installing dependencies...")
print("(This may take 2-3 minutes)\n")

# Core dependencies for ADOFAI training
!pip install -q pydub tqdm

# PyTorch should already be installed in Colab
# but ensure numpy is available
!pip install -q numpy

print("\n✓ Dependencies installed")

## 5. Verify Dataset Loading

In [ ]:
# Test dataset loading
from adofai.dataset import AdofaiDataset
from adofai.tokenizer import AdofaiTokenizer

print("Loading dataset (train split)...")
test_dataset = AdofaiDataset(
    data_dir=DATA_DIR,
    split='train',
    max_samples=5,  # Just load 5 to verify
)

samples = list(test_dataset)
print(f"\n✓ Successfully loaded {len(samples)} samples")

if len(samples) > 0:
    sample = samples[0]
    print(f"\nSample info:")
    print(f"  Chart: {sample['chart_name']}")
    print(f"  Audio shape: {sample['audio'].shape}")
    print(f"  BPM: {sample['bpm']}")
    print(f"  Events: {len(sample['events'])}")

# Initialize tokenizer
print("\nInitializing tokenizer...")
tokenizer = AdofaiTokenizer()
print(f"✓ Tokenizer ready (vocab_size={tokenizer.vocab_size})")

## 6. Smoke Test (Optional)

Run a quick training test with minimal data and a tiny model to verify everything works.

In [ ]:
# Smoke test: quick validation
# This trains on 5 samples for 2 epochs with a tiny model

if SMOKE_MODE or input("Run smoke test? (y/n): ").lower() == 'y':
    print("\n🔥 Running smoke test...\n")
    
    !python -m adofai.train \
        --data_dir {DATA_DIR} \
        --output_dir {OUTPUT_DIR}/smoke \
        --smoke \
        --device {DEVICE}
    
    print("\n✓ Smoke test complete!")
    print(f"  Checkpoints saved to {OUTPUT_DIR}/smoke")
else:
    print("Skipping smoke test")

## 7. Full Training

Train on your full dataset. This will take several hours depending on:
- Dataset size (100+ charts recommended)
- GPU type (T4/P100/V100)
- Number of epochs

**Colab Limits**:
- **Free tier**: ~12 hours per session (disconnect = lost progress)
- **Colab Pro**: Longer sessions, faster GPUs

**Tips**:
- Checkpoints save to Drive after each epoch
- If disconnected, re-run and it will resume from last checkpoint
- Monitor GPU usage: Runtime → Manage sessions

In [ ]:
# Full training
import time

print("Starting full training...")
print(f"  Dataset: {DATA_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Device: {DEVICE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Max samples: {MAX_SAMPLES or 'all'}")
print("\n" + "="*60 + "\n")

start_time = time.time()

# Build command
cmd = f"""python -m adofai.train \
    --data_dir {DATA_DIR} \
    --output_dir {OUTPUT_DIR} \
    --batch_size {BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --epochs {EPOCHS} \
    --device {DEVICE}"""

if MAX_SAMPLES is not None:
    cmd += f" --max_samples {MAX_SAMPLES}"

# Run training
!{cmd}

elapsed = time.time() - start_time
print("\n" + "="*60)
print(f"✓ Training complete!")
print(f"  Time: {elapsed/3600:.2f} hours")
print(f"  Checkpoints: {OUTPUT_DIR}")

## 8. Resume Training (If Interrupted)

If your session disconnects, re-run the setup cells above, then run this cell to resume from the last checkpoint.

In [ ]:
# Resume from checkpoint
import glob

checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint_epoch*.pt"))
if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f"Found checkpoint: {latest_checkpoint}")
    
    # Extract epoch number
    import re
    match = re.search(r'epoch(\d+)', latest_checkpoint)
    if match:
        last_epoch = int(match.group(1))
        remaining_epochs = EPOCHS - last_epoch
        
        print(f"  Last completed epoch: {last_epoch}")
        print(f"  Remaining epochs: {remaining_epochs}")
        
        if remaining_epochs > 0:
            print("\nNote: Current training script doesn't support checkpoint resuming yet.")
            print("      You can restart training or adjust EPOCHS to continue.")
            print("      TODO: Add --checkpoint_path argument to train.py")
        else:
            print("\n✓ Training already complete!")
else:
    print("No checkpoints found. Run the full training cell above.")

## 9. Check Training Logs

View loss curves and checkpoint info.

In [ ]:
# List checkpoints
import glob
import torch

checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint_epoch*.pt"))
print(f"Found {len(checkpoints)} checkpoints:\n")

for ckpt in checkpoints:
    data = torch.load(ckpt, map_location='cpu')
    epoch = data.get('epoch', '?')
    loss = data.get('train_loss', '?')
    print(f"  Epoch {epoch+1}: loss={loss:.4f} - {ckpt}")

## 10. Next Steps

After training:

1. **Download checkpoints** from Google Drive to your local machine
2. **Test inference** with trained model (inference notebook TODO)
3. **Iterate**: Adjust hyperparameters, collect more data, train longer

**For production quality**:
- Integrate Whisper encoder from osuT5 (replace LSTM)
- Use spectrogram features instead of raw audio
- Train on 500+ charts
- Add evaluation metrics

See `docs/ADOFAI.md` in the repo for more details.

---

## Troubleshooting

**Out of Memory (OOM)**:
- Reduce `BATCH_SIZE` (try 2 or 1)
- Use CPU if GPU is too small (slow but works)

**No GPU**:
- Go to Runtime → Change runtime type → GPU (T4)
- Free tier: limited hours per day

**Dataset not found**:
- Check `DATA_DIR` path matches your Drive structure
- Ensure charts are in `<id>__<name>/` subdirectories
- Each chart needs `level.adofai` + audio file

**Training very slow**:
- Verify GPU is enabled (check cell 1)
- Free tier GPUs are slower than Pro
- CPU training is ~10x slower

**Session disconnects**:
- Checkpoints save to Drive after each epoch
- Restart notebook and resume (see cell 8)
- Colab Pro has longer sessions